# Fulltext and Hybrid Search

The previous notebooks used vector search (semantic similarity) as the entry point for retrieval. This notebook introduces a complementary approach: **fulltext keyword search** using Apache Lucene indexes. It then combines both methods for **agent-driven hybrid search**, giving the agent the broadest possible context to reason over.

All queries are executed through MCP using the **Strands Agents SDK**, with custom `@tool` wrappers that encapsulate embedding generation so the agent never sees raw float arrays.

**Learning Objectives:**
- Use fulltext search operators (fuzzy, wildcard, boolean) for keyword-based retrieval
- Understand when fulltext search outperforms vector search and vice versa
- Combine both approaches with custom `@tool` wrappers for agent-driven hybrid search
- Inspect raw tool call inputs and outputs for debugging

**How it works:**
1. Fulltext search uses Apache Lucene indexes for exact keyword matching — no embedding needed
2. Vector search uses embeddings for semantic similarity — finds meaning, not exact words
3. Hybrid search runs both and lets the agent synthesize from combined results
4. Custom `@tool` wrappers encapsulate embedding generation so the agent never sees raw float arrays

**Prerequisites:**
- `../CONFIG.txt` configured with `MCP_GATEWAY_URL` and `MCP_ACCESS_TOKEN`

> The database, embeddings, and indexes are already set up via the pre-deployed MCP server. You do not need to load data yourself.

In [ ]:
%pip install strands-agents strands-agents-tools mcp httpx nest-asyncio -q

## 1. Configuration and Setup

In [ ]:
import os
import nest_asyncio

from dotenv import load_dotenv
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent, tool
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient

nest_asyncio.apply()

from lib.lab_4_data_utils import get_embedding

load_dotenv('../CONFIG.txt')
MODEL_ID = os.getenv('MODEL_ID')
REGION = os.getenv('REGION', 'us-east-1')
MCP_GATEWAY_URL = os.getenv('MCP_GATEWAY_URL')
MCP_ACCESS_TOKEN = os.getenv('MCP_ACCESS_TOKEN')

assert MCP_GATEWAY_URL, "MCP_GATEWAY_URL not configured in CONFIG.txt"
assert MCP_ACCESS_TOKEN, "MCP_ACCESS_TOKEN not configured in CONFIG.txt"

bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature=0,
)

mcp_client = MCPClient(
    lambda: streamablehttp_client(
        url=MCP_GATEWAY_URL,
        headers={"Authorization": f"Bearer {MCP_ACCESS_TOKEN}"},
    )
)

test_emb = get_embedding('test')
print(f'Model:     {MODEL_ID}')
print(f'Embedding: {len(test_emb)} dimensions')
print('Setup complete!')

## 2. Fulltext Search Agent

Fulltext search uses Apache Lucene indexes to match exact keywords in chunk text. Unlike vector search, no embedding is needed — the agent passes a search term directly to the fulltext index.

This is useful for queries with specific entity names, ticker symbols, or technical terms that vector search might not rank highly.

In [ ]:
FULLTEXT_PROMPT = """You are a retrieval assistant that performs fulltext keyword search against a Neo4j database containing SEC 10-K filing data.

You have MCP tools to execute Cypher queries. Use the fulltext index on Chunk text:

CALL db.index.fulltext.queryNodes('search_chunks', $search_term)
YIELD node, score
RETURN node.text AS text, score
ORDER BY score DESC
LIMIT $limit

SEARCH OPERATORS (use these in the search term string):
- Fuzzy: append ~ to handle typos (e.g., 'revnue~' matches 'revenue')
- Wildcard: append * for prefix matching (e.g., 'risk*' matches 'risks', 'risky')
- Boolean AND: both terms required (e.g., 'revenue AND growth')
- Boolean NOT: exclude terms (e.g., 'revenue NOT decline')

For each result, show the Lucene relevance score and a preview of the chunk text."""

print('Fulltext prompt configured.')

## 3. Basic Fulltext Search

Search for chunks containing a specific keyword. No embedding is needed — the agent passes the term directly to the Lucene index.

In [ ]:
def fulltext_search(term: str, limit: int = 5):
    """Run a fulltext keyword search through the MCP agent."""
    print(f'Search term: "{term}"')
    print('-' * 60)

    with mcp_client:
        tools = mcp_client.list_tools_sync()
        agent = Agent(
            model=bedrock_model,
            system_prompt=FULLTEXT_PROMPT,
            tools=tools,
        )
        response = agent(f"Search for chunks containing '{term}'. Use limit={limit}.")
        print(response)
        return response


result = fulltext_search('revenue')

## 4. Search Operators

Fulltext indexes support several operators that vector search cannot replicate:

| Operator | Syntax | Purpose |
|----------|--------|---------|
| Fuzzy | `term~` | Handles typos and spelling variations |
| Wildcard | `term*` | Matches partial terms |
| Boolean AND | `term1 AND term2` | Both terms must appear |
| Boolean NOT | `term1 NOT term2` | First term present, second absent |

In [ ]:
# Fuzzy search — handles typos
result = fulltext_search('revnue~', limit=3)

In [ ]:
# Wildcard search — prefix matching
result = fulltext_search('risk*', limit=3)

In [ ]:
# Boolean AND — both terms must appear
result = fulltext_search('revenue AND growth', limit=3)

## 5. Fulltext + Graph Traversal

Combine fulltext keyword matching with graph traversal to pull in document metadata and connected entities. This is the same enrichment pattern from notebook 02, but applied to fulltext matches instead of vector matches.

In [ ]:
FULLTEXT_GRAPH_PROMPT = """You are a retrieval assistant that performs fulltext search with graph context against a Neo4j database containing SEC 10-K filing data.

When given a search term, use this Cypher to find keyword matches WITH entity context:

CALL db.index.fulltext.queryNodes('search_chunks', $search_term)
YIELD node, score
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (doc)<-[:FILED]-(company:Company)
WITH node, doc, score,
     collect(DISTINCT company.name) AS companies
OPTIONAL MATCH (product:Product)-[:FROM_CHUNK]->(node)
WITH node, doc, score, companies,
     collect(DISTINCT product.name) AS products
RETURN node.text AS text, score, doc.name AS document, companies, products
ORDER BY score DESC
LIMIT $limit

For each result, show the score, document name, matched text, and any connected companies or products."""


def fulltext_graph_search(term: str, limit: int = 5):
    """Run fulltext search with graph traversal."""
    print(f'Search term: "{term}" (with graph context)')
    print('-' * 60)

    with mcp_client:
        tools = mcp_client.list_tools_sync()
        agent = Agent(
            model=bedrock_model,
            system_prompt=FULLTEXT_GRAPH_PROMPT,
            tools=tools,
        )
        response = agent(f"Search for chunks containing '{term}' with graph context. Use limit={limit}.")
        print(response)
        return response


result = fulltext_graph_search('iPhone')

## 6. Vector vs Fulltext — When to Use Each

| Query Type | Fulltext | Vector |
|------------|----------|--------|
| Specific company name ("Apple Inc") | Strong exact match | May match other companies with similar descriptions |
| Ticker symbol ("AAPL") | Exact keyword match | Embedding may not capture ticker semantics |
| Conceptual question ("supply chain risks") | Only if exact words appear | Finds semantically related content regardless of wording |
| Fuzzy or partial terms ("revnue~") | Built-in fuzzy matching | No fuzzy capability |

Neither approach is strictly better. The next section combines both.

## 7. Agent-Driven Hybrid Search with `@tool` Wrappers

Instead of serializing a 1024-dimensional embedding array into the agent's prompt, we wrap embedding generation and MCP query execution inside custom `@tool` functions. The agent calls `vector_search("Apple products")` — a clean interface that hides the embedding internals.

This section opens a persistent `MCPClient` connection and uses `call_tool_sync()` inside the `@tool` wrappers to execute Cypher queries directly — the same Strands MCP integration used earlier, but invoked programmatically rather than through the agent loop.

In [ ]:
# Open a persistent MCP connection for @tool wrappers
persistent_mcp = MCPClient(lambda: streamablehttp_client(
    url=MCP_GATEWAY_URL,
    headers={"Authorization": f"Bearer {MCP_ACCESS_TOKEN}"},
))
persistent_mcp.__enter__()

# Discover available tools and find the Cypher query tool
mcp_tools = persistent_mcp.list_tools_sync()
tool_names = [t.tool_name for t in mcp_tools]
print(f"MCP tools discovered: {tool_names}")

cypher_tool = next((n for n in tool_names if "read-cypher" in n), None)
assert cypher_tool, f"No Cypher query tool found among: {tool_names}"
print(f"Cypher tool: {cypher_tool}")

In [ ]:
@tool
def vector_search(query: str, top_k: int = 5) -> str:
    """Search for semantically similar chunks using vector embeddings.
    Use this for conceptual or semantic queries where exact words may differ."""
    embedding = get_embedding(query)
    top_k = int(top_k)

    cypher = f"""
        CALL db.index.vector.queryNodes('chunkEmbeddings', {top_k}, $query_vector)
        YIELD node, score
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
        WITH node {{.*, embedding: null}} AS node, score, doc
        RETURN node.text AS text, score, doc.name AS document
        ORDER BY score DESC
    """

    result = persistent_mcp.call_tool_sync(
        tool_use_id="vector-search",
        name=cypher_tool,
        arguments={"query": cypher, "params": {"query_vector": embedding}},
    )
    return result["content"][0]["text"]


@tool
def fulltext_search_tool(term: str, limit: int = 5) -> str:
    """Search for chunks containing specific keywords.
    Use this for exact terms, company names, tickers, or partial matches.
    Supports operators: fuzzy (term~), wildcard (term*), AND, NOT."""
    safe_term = term.replace("\\", "\\\\").replace("'", "\\'")
    limit = int(limit)

    cypher = f"""
        CALL db.index.fulltext.queryNodes('search_chunks', '{safe_term}')
        YIELD node, score
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
        RETURN node.text AS text, score, doc.name AS document
        ORDER BY score DESC
        LIMIT {limit}
    """

    result = persistent_mcp.call_tool_sync(
        tool_use_id="fulltext-search",
        name=cypher_tool,
        arguments={"query": cypher},
    )
    return result["content"][0]["text"]


print('Custom tools created:')
print('  - vector_search')
print('  - fulltext_search_tool')

In [ ]:
HYBRID_PROMPT = """You are a financial analysis assistant that combines vector (semantic) and fulltext (keyword) search to answer questions about SEC 10-K filings.

You have two search tools:
1. vector_search: Finds semantically similar content (good for conceptual questions)
2. fulltext_search_tool: Finds exact keyword matches (good for specific names, terms, tickers)

HYBRID SEARCH STRATEGY:
For comprehensive results, run BOTH search tools with the same query, then synthesize an answer from the combined results. This gives you the benefits of both semantic understanding and keyword precision.

When the query contains specific names or terms (like "Apple", "AAPL"), fulltext search may find more precise matches. When the query is conceptual ("supply chain risks"), vector search captures semantic meaning.

For the best results, always run both searches and compare what each found."""

hybrid_agent = Agent(
    model=bedrock_model,
    system_prompt=HYBRID_PROMPT,
    tools=[vector_search, fulltext_search_tool],
)
print('Hybrid agent ready!')

In [ ]:
async def hybrid_search(query: str):
    """Run hybrid search using both vector and fulltext tools."""
    print(f'Question: "{query}"')
    print('=' * 60)

    response = await hybrid_agent.invoke_async(
        f"Answer this question using BOTH vector search and fulltext search: {query}"
    )
    print(f'\n{response}')
    return response


result = await hybrid_search("What are Apple's key risk factors?")

In [ ]:
result = await hybrid_search('Which companies face cybersecurity-related risks?')

## 8. Understanding Alpha Tuning

The `HybridRetriever` in the neo4j-graphrag library (Lab 5) uses an **alpha** parameter to mathematically balance vector and fulltext scores:

```
combined_score = alpha * vector_score + (1 - alpha) * fulltext_score
```

| Alpha | Behavior |
|-------|----------|
| 1.0 | Pure vector (semantic only) |
| 0.7 | Favor semantic, supplement with keywords |
| 0.5 | Equal weight to both |
| 0.3 | Favor keywords, supplement with semantic |
| 0.0 | Pure fulltext (keywords only) |

**Through MCP, the agent approximates this by running both searches and using its judgment to weigh the results.** This is less mathematically precise than the library approach, but it teaches the underlying concept: different queries benefit from different balances of semantic and keyword matching.

For programmatic alpha tuning with automatic score normalization, see Lab 5's `HybridRetriever` and `HybridCypherRetriever`.

## Summary

You built three search approaches across the Lab 4 notebooks:

| Notebook | Search Method | Graph Context | Embedding Handling |
|----------|--------------|---------------|-------------------|
| 01 | Vector | No | `@tool` wrapper (hidden) |
| 02 | Vector + Graph Traversal | Document + Chunks + Entities | `@tool` wrapper (hidden) |
| 03 | Fulltext | Optional | No embedding needed |
| 03 | Hybrid (`@tool`) | Optional | `@tool` wrapper (hidden) |

The progression shows what makes searching a knowledge graph different from a standalone vector database: graph-enriched search follows relationships to connected entities, turning text retrieval into knowledge retrieval. The `@tool` wrapper pattern keeps embedding generation on the data plane so the agent works with clean string interfaces.

For the programmatic `HybridRetriever` and `HybridCypherRetriever` classes with automatic alpha tuning, see [Lab 5](../Lab_5_GraphRAG/).